# Evaluate a pretrained gas classifier

The four validated checkpoints are included in this repository. Install the environment,
extract the dataset supplied with the paper beside this notebook, choose a config below, and run all cells.
No model training is needed. Accuracies print before the confusion matrix. For chemical
humidity, four lines report RH 0%, RH 30%, RH 50%, and all held-out samples combined.
The confusion matrix includes all held-out samples, including all three RH conditions.


In [ ]:
# Select one of the four included pretrained models.
CONFIG_NAME = "Config_TBDT_Chem_dry_pretrained"
# Keep "pretrained" to use the included model, or enter your own saved run folder.
RUN_FOLDER_NAME = "pretrained"

# Available choices:
# "Config_TBDT_Chem_dry_pretrained"
# "Config_TBDT_Chem_humid_pretrained"
# "Config_TBDT_Solvent_pretrained"
# "Config_TBDT_Bacteria_pretrained"

from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import ML_models
ML_models = importlib.reload(ML_models)
from configs import load_config
from src import Data_pipeline, Run_analysis
Data_pipeline = importlib.reload(Data_pipeline)
Run_analysis = importlib.reload(Run_analysis)

CONFIG = load_config(CONFIG_NAME)
OUTPUT = CONFIG["OUTPUT"]
if not RUN_FOLDER_NAME.strip() or RUN_FOLDER_NAME.strip().lower() == "latest":
    raise ValueError("Set RUN_FOLDER_NAME to pretrained or the exact name of your saved run.")
runs_root = PROJECT_ROOT / OUTPUT["models_dir"] / OUTPUT["runs_dirname"]
run_dir = Run_analysis.resolve_run_dir(RUN_FOLDER_NAME, runs_root)
run_metadata = Run_analysis.load_run_metadata(run_dir)
RUN_CONFIG = run_metadata["config"]
if RUN_CONFIG.get("CONFIG_NAME") != CONFIG_NAME:
    raise ValueError("RUN_FOLDER_NAME does not belong to the selected CONFIG_NAME.")
print("Selected config:", CONFIG_NAME)
print("Selected run folder:", RUN_FOLDER_NAME)


In [ ]:
dataset_config = RUN_CONFIG["DATASET"]
preprocess_config = RUN_CONFIG["PREPROCESS"]
selection_config = RUN_CONFIG["SELECTION"]
data_root = Data_pipeline.resolve_data_root(dataset_config["data_dir"])
data_bundle = Data_pipeline.prepare_data_bundle(
    data_root=data_root,
    train_rh_values=selection_config["train_rh_values"],
    heldout_test_rh_values=selection_config["heldout_test_rh_values"],
    average_mode=preprocess_config["average_mode"],
    cut_window=preprocess_config["cut_window"],
    val_size=selection_config["train_val_split"]["val_size"],
    random_state=selection_config["train_val_split"]["random_state"],
)
if data_bundle["gas_label_map"] != run_metadata["gas_label_map"]:
    raise ValueError("Current data labels do not match the selected run.")
device = "cuda" if torch.cuda.is_available() else "cpu"
model, loaded_model_metadata = Run_analysis.load_model(run_dir, device=device)
heldout_evaluation = Run_analysis.evaluate_heldout(
    model, data_bundle, device=device,
    batch_size=RUN_CONFIG["TRAINING"]["batch_size"],
)
plt.show()


In [ ]:
output_dir = PROJECT_ROOT / "figures" / "confusion_matrices" / CONFIG_NAME / RUN_FOLDER_NAME
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "heldout_test_confusion_matrix.png"
heldout_evaluation["figure"].savefig(output_path, dpi=300, bbox_inches="tight")
print("Saved figure:", output_path)
